ETAPA 1 — Preparar carpetas, bibliotecas y parámetros

In [48]:
from pathlib import Path
import pandas as pd
import requests

BASE = Path.cwd()  # Carpeta en la que ejecutas VS Code
ENTREGA = BASE / "03_entrega"
ENTREGA.mkdir(exist_ok=True)

PROVINCIAS = {
    "PIURA": (-5.1945, -80.6328),
    "SULLANA": (-4.9039, -80.6853),
    "MORROPON": (-5.1822, -79.9690),
    "PAITA": (-5.0852, -81.1137)
}
INICIO = "20000101"
FIN = "20261020"  # Fecha final del análisis original (no es una descarga en tiempo real)

CLIMA_CSV = BASE / "clima_cuatro_provincias_diario.csv"
print("Carpeta de trabajo:", BASE)
print("Salida:", ENTREGA)


Carpeta de trabajo: c:\Users\veren\OneDrive\Desktop\Verenisse\Cayetano\ciclos\6to\hackathon
Salida: c:\Users\veren\OneDrive\Desktop\Verenisse\Cayetano\ciclos\6to\hackathon\03_entrega


ETAPA 2 — Descargar o reutilizar NASA POWER 
Datos que obtenemos: temperatura máxima (T2M_MAX), temperatura mínima (T2M_MIN) y precipitación corregida (PRECTOTCORR), diariamente entre 2000 y 20/10/2026.

Estos son puntos de referencia cercanos a las ciudades; no representan promedios de toda la provincia.

In [ ]:
def descargar_nasa(provincia, latitud, longitud):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    parametros = {
        "parameters": "T2M_MAX,T2M_MIN,PRECTOTCORR",
        "community": "AG",
        "longitude": longitud,
        "latitude": latitud,
        "start": INICIO,
        "end": FIN,
        "format": "JSON"
    }
    respuesta = requests.get(url, params=parametros, timeout=120)
    respuesta.raise_for_status()
    v = respuesta.json()["properties"]["parameter"]

    fechas = sorted(v["T2M_MAX"])
    datos = pd.DataFrame({
        "provincia": provincia,
        "fecha": pd.to_datetime(fechas, format="%Y%m%d"),
        "tmax": [v["T2M_MAX"].get(d) for d in fechas],
        "tmin": [v["T2M_MIN"].get(d) for d in fechas],
        "precipitacion": [v["PRECTOTCORR"].get(d) for d in fechas]
    })
    cols = ["tmax","tmin","precipitacion"]
    datos[cols] = datos[cols].mask(datos[cols] <= -900)
    return datos

if CLIMA_CSV.exists():
    print("Ya existe la base climática. La reutilizaremos (sin descargar).")
    clima_diario = pd.read_csv(CLIMA_CSV, parse_dates=["fecha"])
else:
    print("Descargando NASA POWER para cuatro puntos de referencia...")
    tablas_clima = []
    for provincia, (lat, lon) in PROVINCIAS.items():
        tabla = descargar_nasa(provincia, lat, lon)
        tablas_clima.append(tabla)
        print(provincia, "→", len(tabla), "días")
    clima_diario = pd.concat(tablas_clima, ignore_index=True)
    clima_diario = clima_diario.sort_values(["provincia","fecha"])
    clima_diario.to_csv(CLIMA_CSV, index=False)
    print("Creado:", CLIMA_CSV.name)

print("Total de registros diarios:", len(clima_diario))
print(clima_diario.groupby("provincia")["fecha"].agg(["min","max","count"]))
display(clima_diario.head())


✅ Ya existe la base climática. La reutilizaremos (sin descargar).
Total de registros diarios: 39160
                 min        max  count
provincia                             
MORROPON  2000-01-01 2026-10-20   9790
PAITA     2000-01-01 2026-10-20   9790
PIURA     2000-01-01 2026-10-20   9790
SULLANA   2000-01-01 2026-10-20   9790


,fecha,tmax,tmin,precipitacion,provincia
0,2000-01-01,28.20,14.57,0.02,MORROPON
1,2000-01-02,28.15,15.08,0.00,MORROPON
2,2000-01-03,28.65,14.34,0.01,MORROPON
3,2000-01-04,25.97,14.74,0.00,MORROPON
4,2000-01-05,26.10,14.86,0.00,MORROPON


ETAPA 3 — Comprobar la calidad de NASA POWER
Comprobamos duplicados, valores faltantes y si NASA POWER asignó las mismas mediciones a Piura y Sullana por resolución espacial.

In [50]:
columnas_clima = ["tmax","tmin","precipitacion"]
faltantes_clima = clima_diario.groupby("provincia")[columnas_clima].apply(lambda x:x.isna().sum())
duplicados_clima = clima_diario.duplicated(["provincia","fecha"]).sum()

print("VALORES FALTANTES:")
display(faltantes_clima)
print("Registros provincia-fecha duplicados:", duplicados_clima)

p = clima_diario.loc[clima_diario["provincia"] == "PIURA"].set_index("fecha")
s = clima_diario.loc[clima_diario["provincia"] == "SULLANA"].set_index("fecha")
print("\n¿Piura y Sullana tienen clima idéntico?")
for var in columnas_clima:
    print(var, ":", p[var].equals(s[var]))

if duplicados_clima:
    raise ValueError("Hay duplicados climáticos: revisar antes de integrar.")


VALORES FALTANTES:


,tmax,tmin,precipitacion
provincia,,,
MORROPON,0,0,0
PAITA,0,0,0
PIURA,0,0,0
SULLANA,0,0,0


Registros provincia-fecha duplicados: 0

¿Piura y Sullana tienen clima idéntico?
tmax : True
tmin : True
precipitacion : True


ETAPA 4 — Convertir datos climáticos DIARIOS a SEMANALES
Agrupamos de domingo a sábado.
- tmax_promedio: media de temperaturas máximas diarias.
- tmin_promedio: media de temperaturas mínimas diarias.
- lluvia_total: suma de precipitaciones diarias.
Exigimos 7 fechas y 7 observaciones válidas para considerar completa una semana. La definición exacta de semana epidemiológica oficial todavía debe verificarse históricamente.

In [51]:
clima = clima_diario.copy()
clima["fecha"] = pd.to_datetime(clima["fecha"])
clima["inicio_semana"] = clima["fecha"] - pd.to_timedelta(
    (clima["fecha"].dt.dayofweek + 1) % 7, unit="D"
)
clima_semanal = (clima.groupby(["provincia","inicio_semana"])
    .agg(tmax_promedio=("tmax","mean"),
         tmin_promedio=("tmin","mean"),
         lluvia_total=("precipitacion","sum"),
         dias=("fecha","nunique"),
         dias_tmax=("tmax","count"),
         dias_tmin=("tmin","count"),
         dias_lluvia=("precipitacion","count")).reset_index())
clima_semanal["semana_completa"] = (
    clima_semanal["dias"].eq(7)
    & clima_semanal[["dias_tmax","dias_tmin","dias_lluvia"]].eq(7).all(axis=1)
)
print("Filas climáticas semanales:", len(clima_semanal))
print("Semanas incompletas por provincia:")
print(clima_semanal.groupby("provincia")["semana_completa"].apply(lambda x:(~x).sum()))
display(clima_semanal.head())



Filas climáticas semanales: 5600
Semanas incompletas por provincia:
provincia
MORROPON    2
PAITA       2
PIURA       2
SULLANA     2
Name: semana_completa, dtype: int64


C:\Users\veren\AppData\Local\Temp\ipykernel_24468\1597510329.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  (clima["fecha"].dt.dayofweek + 1) % 7, unit="D"


,provincia,inicio_semana,tmax_promedio,tmin_promedio,lluvia_total,dias,dias_tmax,dias_tmin,dias_lluvia,semana_completa
0,MORROPON,1999-12-26,28.200000,14.570000,0.02,1,1,1,1,False
1,MORROPON,2000-01-02,27.958571,14.915714,0.02,7,7,7,7,True
2,MORROPON,2000-01-09,28.915714,15.564286,4.78,7,7,7,7,True
3,MORROPON,2000-01-16,29.718571,16.200000,3.26,7,7,7,7,True
4,MORROPON,2000-01-23,28.161429,17.318571,11.85,7,7,7,7,True


ETAPA 5 — Leer los cuatro archivos de dengue de tu compañero
No descargamos ni reconstruimos el CDC: usamos los cuatro archivos procesados que ya tienes. Los códigos A97.0C, A97.1C y A97.2C se suman como confirmados; los terminados en P, como probables. No los mezclamos.

⚠️ En estos archivos pueden existir semanas rellenadas con cero automáticamente. Eso no demuestra que el dengue haya estado ausente.

In [52]:
columnas_confirmados = ["A97.0C","A97.1C","A97.2C"]
columnas_probables = ["A97.0P","A97.1P","A97.2P"]
tablas = []
for provincia in PROVINCIAS:
    archivo = BASE / f"PIURA_{provincia}.csv"
    if not archivo.exists():
        raise FileNotFoundError(f"Falta {archivo.name}. Colócalo junto a este notebook.")
    df = pd.read_csv(archivo)
    df["provincia"] = provincia
    df["casos_confirmados"] = df[columnas_confirmados].sum(axis=1)
    df["casos_probables"] = df[columnas_probables].sum(axis=1)
    tablas.append(df[["provincia","ano","semana","casos_confirmados","casos_probables"]])

dengue_semanal = pd.concat(tablas, ignore_index=True)
if dengue_semanal.duplicated(["provincia","ano","semana"]).any():
    raise ValueError("Hay provincia-año-semana repetidos en dengue.")
print("Filas de dengue:", len(dengue_semanal))
display(dengue_semanal.groupby("provincia")[["casos_confirmados","casos_probables"]].sum())



Filas de dengue: 5459


,casos_confirmados,casos_probables
provincia,,
MORROPON,23871,1211
PAITA,15170,92
PIURA,108074,5908
SULLANA,40358,10563


ETAPA 6 — Relacionar dengue y clima
Se aplica la misma regla provisional de semana domingo–sábado que contiene el 4 de enero = semana 1.

Precaución importante: hay un caso confirmado de Sullana, 2009, semana 53 que esta regla excluye. No lo reasignamos a 2010 sin verificar el calendario oficial. Los casos excluidos con conteos positivos se guardan en archivo separado.

In [54]:
def inicio_ano_epi_provisional(ano):
    enero4 = pd.Timestamp(year=int(ano), month=1, day=4)
    return enero4 - pd.Timedelta(days=(enero4.day_of_week + 1) % 7)

dengue = dengue_semanal.copy()
dengue["inicio_semana"] = [
    inicio_ano_epi_provisional(ano) + pd.Timedelta(weeks=int(semana)-1)
    for ano, semana in zip(dengue["ano"], dengue["semana"])
]
dengue["semanas_del_ano"] = dengue["ano"].apply(
    lambda ano:(inicio_ano_epi_provisional(ano+1)-inicio_ano_epi_provisional(ano)).days//7
)
dengue["semana_valida"] = dengue["semana"].ge(1) & dengue["semana"].le(dengue["semanas_del_ano"])
descartadas = dengue.loc[~dengue["semana_valida"]].copy()
pendientes = descartadas.loc[
    descartadas["casos_confirmados"].gt(0) | descartadas["casos_probables"].gt(0)
].copy()
unido = dengue.loc[dengue["semana_valida"]].merge(
    clima_semanal, on=["provincia","inicio_semana"], how="left",validate="one_to_one")
integrado = unido.loc[unido["semana_completa"].eq(True)].copy()

print("Semanas descartadas por la regla provisional:",len(descartadas))
print("Casos CONFIRMADOS pendientes:",pendientes["casos_confirmados"].sum())
print("Casos PROBABLES pendientes:",pendientes["casos_probables"].sum())
print("Semanas sin clima completo:",len(unido)-len(integrado))
print("Filas integradas:",len(integrado))
display(pendientes[["provincia","ano","semana","casos_confirmados","casos_probables"]])


Semanas descartadas por la regla provisional: 83
Casos CONFIRMADOS pendientes: 1
Casos PROBABLES pendientes: 0
Semanas sin clima completo: 0
Filas integradas: 5376


,provincia,ano,semana,casos_confirmados,casos_probables
1907,SULLANA,2009,53,1,0


ETAPA 7 — Auditoría y tres archivos de salida
Crearemos solo tres CSV de entrega en la carpeta 03_entrega, sin generar archivos de prueba adicionales.

| Archivo | Uso |
|---|---|
| `dengueshield_dataset_integrado_REVISION.csv` | Principal: datos de clima + dengue semanales |
| `dengueshield_reporte_calidad.csv` | Resumen por año/provincia, incluyendo años con cero casos confirmados registrados |
| `dengueshield_casos_pendientes_revision.csv` | Casos positivos excluidos por la regla de calendario provisional |

El CSV climático diario clima_cuatro_provincias_diario.csv también es importante como dato fuente y se conserva en la carpeta principal.

In [59]:
integrado["anio_sin_casos_registrados"] = (
    integrado.groupby(["provincia","ano"])["casos_confirmados"].transform("sum").eq(0)
)
integrado = integrado.sort_values(["provincia","ano","semana"]).reset_index(drop=True)
reporte = (integrado.groupby(["provincia","ano"])
    .agg(semanas=("semana","nunique"),
         casos_confirmados=("casos_confirmados","sum"),
         casos_probables=("casos_probables","sum"),
         semanas_con_confirmados=("casos_confirmados",lambda x:(x>0).sum()),
         anio_sin_casos_registrados=("anio_sin_casos_registrados","first")).reset_index())

columnas_finales = [
    "provincia","ano","semana","inicio_semana",
    "casos_confirmados","casos_probables",
    "tmax_promedio","tmin_promedio","lluvia_total",
    "anio_sin_casos_registrados"
]
r1 = ENTREGA/"dengueshield_dataset_integrado_REVISION.csv"
r2 = ENTREGA/"dengueshield_reporte_calidad.csv"
r3 = ENTREGA/"dengueshield_casos_pendientes_revision.csv"

integrado[columnas_finales].to_csv(r1,index=False)
reporte.to_csv(r2,index=False)
pendientes.to_csv(r3,index=False)

print("Archivos de entrega:")
for r in [r1,r2,r3]:
    print(" •",r.name)
print("\nAVISO: integración PROVISIONAL; calendario y ceros requieren validación.")
display(integrado[columnas_finales].head())



Archivos de entrega:
 • dengueshield_dataset_integrado_REVISION.csv
 • dengueshield_reporte_calidad.csv
 • dengueshield_casos_pendientes_revision.csv

AVISO: integración PROVISIONAL; calendario y ceros requieren validación.


,provincia,ano,semana,inicio_semana,casos_confirmados,casos_probables,tmax_promedio,tmin_promedio,lluvia_total,anio_sin_casos_registrados
0,MORROPON,2000,1,2000-01-02,0,1,27.958571,14.915714,0.02,False
1,MORROPON,2000,2,2000-01-09,0,0,28.915714,15.564286,4.78,False
2,MORROPON,2000,3,2000-01-16,0,0,29.718571,16.200000,3.26,False
3,MORROPON,2000,4,2000-01-23,0,0,28.161429,17.318571,11.85,False
4,MORROPON,2000,5,2000-01-30,0,0,28.225714,17.062857,16.93,False
